# Learned Stage-Wise Curvature Tuning on FGVC Aircraft

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the FGVC-Aircraft dataset using an ImageNet-pretrained ResNet-18.

FGVC-Aircraft is a fine-grained image-classification dataset in which the model must distinguish between visually similar aircraft categories. This makes it useful for testing whether stage-wise curvature adaptation can help when the downstream task requires subtle feature discrimination.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization changes accuracy or the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Thu Oct  1 14:36:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 5.70 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 43.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 128.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 111.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 16.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [1]:
from pathlib import Path

data_file = Path(
    "/content/curvature-tuning-research/src/curvature-tuning/utils/data.py"
)

text = data_file.read_text(encoding="utf-8")

old = '''    elif dataset_to_use == 'fgvc-aircraft':
        hf_dataset = datasets.load_dataset(
            "HuggingFaceM4/FGVC-Aircraft",
            num_proc=2
        )

        hf_trainset = hf_dataset["train"]
        hf_valset = hf_dataset["validation"]
        hf_testset = hf_dataset["test"]

        train_set = HuggingFaceDataset(
            hf_trainset,
            transform=transform_train
        )

        val_set = HuggingFaceDataset(
            hf_valset,
            transform=transform_test
        )

        test_set = HuggingFaceDataset(
            hf_testset,
            transform=transform_test
        )
'''

new = '''    elif dataset_to_use == 'fgvc-aircraft':
        train_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="train",
            annotation_level="variant",
            transform=transform_train,
            download=True,
        )

        val_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="val",
            annotation_level="variant",
            transform=transform_test,
            download=True,
        )

        test_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="test",
            annotation_level="variant",
            transform=transform_test,
            download=True,
        )
'''

if new in text:
    print("FGVC-Aircraft patch already applied.")
elif old in text:
    text = text.replace(old, new)
    data_file.write_text(text, encoding="utf-8")
    print("FGVC-Aircraft soft patch applied.")
else:
    print("Expected FGVC-Aircraft block was not found. Nothing changed.")

FGVC-Aircraft soft patch applied.


In [2]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


## FGVC Aircraft Dataset

Before running the experiments, the FGVC Aircraft data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [3]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_fgvc-aircraft",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

100%|██████████| 2.75G/2.75G [03:08<00:00, 14.6MB/s] 


Train samples: 3334
Validation samples: 3333
Test samples: 3333
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([71, 97, 65, 65, 24, 48, 84, 74, 38, 95])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, FGVC Aircraft dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 08:22:59.569 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs20.log
2026-10-01 08:22:59.576 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 189MB/s]
2026-10-01 08:23:00.492 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 08:23:00.516 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 08:23:00.516 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 08:24:26.740 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.710100, Accuracy: 0.00%
2026-10-01 08:24:27.025 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.646141, Accuracy: 2.17%
2026-10-01 08:24:27.231 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.324793, Val Accuracy: 6.54%
2026-10-01 08:24:27.23

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 09:17:02.571 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed43_epochs20.log
2026-10-01 09:17:02.575 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 09:17:03.108 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 09:17:03.126 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 09:17:03.126 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 09:18:29.117 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.621954, Accuracy: 0.00%
2026-10-01 09:18:29.386 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.632243, Accuracy: 1.89%
2026-10-01 09:18:29.574 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.359454, Val Accuracy: 4.68%
2026-10-01 09:18:29.575 | INFO     | train:linear_probe:190 - New best validation accuracy: 4.68 at epoch 1
2026-10-01 09:18:29.621 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.28

In [6]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 10:10:25.844 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed44_epochs20.log
2026-10-01 10:10:25.847 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 10:10:26.371 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 10:10:26.386 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 10:10:26.386 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 10:11:53.068 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.014167, Accuracy: 0.00%
2026-10-01 10:11:53.327 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.619449, Accuracy: 2.48%
2026-10-01 10:11:53.520 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.262840, Val Accuracy: 8.04%
2026-10-01 10:11:53.521 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.04 at epoch 1
2026-10-01 10:11:53.573 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.12

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 11:03:49.496 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 11:03:49.589 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:03:50.206 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 11:03:50.207 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:03:50.241 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 11:03:50.242 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:03:50.242 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 11:03:53.090 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.867596, Accuracy: 0.00%
2026-10-01 11:04:35.965 | INFO     | train:train_epoch

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 11:34:36.771 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 11:34:36.829 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:34:37.299 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 11:34:37.299 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:34:37.325 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 11:34:37.325 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:34:37.326 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 11:34:39.228 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.624043, Accuracy: 0.00%
2026-10-01 11:35:23.508 | INFO     | train:train_epoch

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 12:05:18.942 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 12:05:19.001 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 12:05:19.455 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 12:05:19.455 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 12:05:19.477 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 12:05:19.478 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 12:05:19.478 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 12:05:21.251 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.809299, Accuracy: 3.12%
2026-10-01 12:06:03.922 | INFO     | train:train_epoch

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [4]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 13:42:15.259 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-01 13:42:15.327 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 169MB/s]
2026-10-01 13:42:16.157 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 13:42:16.157 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 13:42:16.387 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 13:42:16.388 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 13:42:16.388 | INFO     | __main__:main:439 - Total traina

In [4]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 14:44:07.133 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-01 14:44:07.201 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 166MB/s]
2026-10-01 14:44:08.077 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 14:44:08.077 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 14:44:08.180 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 14:44:08.181 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 14:44:08.181 | INFO     | __main__:main:439 - Total trai

In [5]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 15:15:20.662 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-01 15:15:20.723 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 15:15:21.184 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 15:15:21.184 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 15:15:21.214 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 15:15:21.215 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 15:15:21.215 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 15:15:23.224 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.799154, Accuracy: 0.00%
2026-10-01 15:16:07.142 | INFO     | tra

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [6]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 15:46:50.413 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs30.log
2026-10-01 15:46:50.417 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 15:46:50.943 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 15:46:50.958 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 15:46:50.959 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 15:48:18.917 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.710100, Accuracy: 0.00%
2026-10-01 15:48:19.302 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.646141, Accuracy: 2.17%
2026-10-01 15:48:19.616 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.324793, Val Accuracy: 6.54%
2026-10-01 15:48:19.617 | INFO     | train:linear_probe:190 - New best validation accuracy: 6.54 at epoch 1
2026-10-01 15:48:19.701 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.06

In [7]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 16:44:52.056 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed43_epochs30.log
2026-10-01 16:44:52.060 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 16:44:52.794 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 16:44:52.821 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 16:44:52.821 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 16:46:20.186 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.621954, Accuracy: 0.00%
2026-10-01 16:46:20.470 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.632243, Accuracy: 1.89%
2026-10-01 16:46:20.667 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.359454, Val Accuracy: 4.68%
2026-10-01 16:46:20.668 | INFO     | train:linear_probe:190 - New best validation accuracy: 4.68 at epoch 1
2026-10-01 16:46:20.721 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.28

In [8]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 17:42:07.035 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet18_seed44_epochs30.log
2026-10-01 17:42:07.038 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 17:42:07.559 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 17:42:07.574 | INFO     | __main__:main:116 - Number of trainable parameters: 51300
2026-10-01 17:42:07.574 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 17:43:34.109 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.014167, Accuracy: 0.00%
2026-10-01 17:43:34.385 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.619449, Accuracy: 2.48%
2026-10-01 17:43:34.581 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.262840, Val Accuracy: 8.04%
2026-10-01 17:43:34.581 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.04 at epoch 1
2026-10-01 17:43:34.634 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.12

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 18:39:17.700 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 18:39:17.758 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 18:39:18.213 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 18:39:18.213 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 18:39:18.236 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 18:39:18.237 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 18:39:18.237 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 18:39:20.184 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.867596, Accuracy: 0.00%
2026-10-01 18:40:03.080 | INFO     | train:train_epoch

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 19:25:01.323 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 19:25:01.411 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 19:25:02.078 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 19:25:02.078 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 19:25:02.113 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 19:25:02.114 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 19:25:02.114 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 19:25:04.123 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.624043, Accuracy: 0.00%
2026-10-01 19:25:48.040 | INFO     | train:train_epoch

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 20:11:41.604 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 20:11:41.672 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 20:11:42.164 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 20:11:42.165 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 20:11:42.192 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 20:11:42.193 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 20:11:42.193 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 20:11:44.310 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.809299, Accuracy: 3.12%
2026-10-01 20:12:34.142 | INFO     | train:train_epoch

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 21:04:57.740 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 21:04:57.806 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 21:04:58.295 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 21:04:58.295 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 21:04:58.321 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 21:04:58.322 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 21:04:58.322 | INFO     | __main__:main:439 - Total trainable parameters: 51304
2026-10-01 21:05:00.342 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.883708, Accuracy: 0.00%
2026-10-01 21:05:49.293 | INFO     | train

: 

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30